<a href="https://colab.research.google.com/github/flipiwolker-alt/cv-video-analytics/blob/main/notebooks/PZ_4_Whisper_Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ПЗ 4 — Извлечение аудио и распознавание речи (Whisper)

Вытаскиваем аудиодорожку из видео через moviepy, прогоняем через Whisper, получаем текст с таймкодами.

In [1]:
!pip install moviepy openai-whisper -q
!apt-get install -y ffmpeg -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
videos = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mkv'))]
print(f'видео в drive: {videos}')
video_path = f'{VIDEO_DIR}/{videos[0]}'

Mounted at /content/drive
видео в drive: ['video.mp4']


In [3]:
from moviepy.editor import VideoFileClip

audio_path = '/content/audio.wav'
clip = VideoFileClip(video_path)
clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
clip.close()
print('аудио извлечено')

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



аудио извлечено


In [6]:
import whisper

model = whisper.load_model('base')
result = model.transcribe(audio_path, verbose=False)

print('транскрипция готова')
print(result['text'][:500])

  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Detected language: English


100%|██████████| 21901/21901 [01:04<00:00, 342.03frames/s]

транскрипция готова
 You're not strangers to love, you know the rules and something like that I've still committed my struggle to kill her You wouldn't get this wrong any other guy I just wanna tell you how I feel I gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around it He's hurt you Never gonna make you cry Never gonna say goodbye Never gonna tell the lie And hurt you We've known each other for so long Your heart's been aching but You're just trying to say it It's up w


In [7]:
import pandas as pd

rows = [
    {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
    for s in result['segments']
]

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/whisper_transcript.csv', index=False, encoding='utf-8-sig')
print(f'сегментов: {len(df)}')
df.head(10)

сегментов: 67


,start,end,text
0,0.00,26.36,"You're not strangers to love, you know the rul..."
1,26.36,30.36,I've still committed my struggle to kill her
2,30.36,34.36,You wouldn't get this wrong any other guy
3,34.36,38.36,I just wanna tell you how I feel
4,38.36,42.36,I gotta make you understand
5,42.36,44.36,Never gonna give you up
6,44.36,46.36,Never gonna let you down
7,46.36,48.36,Never gonna run around it
8,48.36,50.36,He's hurt you
9,50.36,52.36,Never gonna make you cry
